In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# -----------------------------
# 1) Колонки
# -----------------------------
feat_names = [
    #"linspace",
    "linspace_session_id",
    "linspace_within_session",
    #"brightness",#"time_reaction",
    #"symbol_m", "symbol_c", "symbol_s", "symbol_y", "symbol_f", "symbol_j",
    "location_0", "location_1", "location_2", "location_3",
    "participant_J", "participant_F", "participant_Y",
    #"correct_symbol_m", "correct_symbol_c", "correct_symbol_s", "correct_symbol_y", "correct_symbol_f", "correct_symbol_j",
]
latent_dim = 16
feat_names_fs = [f"f{i}" for i in range(latent_dim)]
feat_names.extend(feat_names_fs)

F_DIM = len(feat_names)

pj = feat_names.index("participant_J")
pf = feat_names.index("participant_F")
py = feat_names.index("participant_Y")
participant_idx = (pj, pf, py)

def cols(prefix: str):
    return [f"{prefix}_{f}" for f in feat_names]

x1_cols = cols("x1")
x2_cols = cols("x2")
x3_cols = cols("x3")
x4_cols = cols("x4")


def prepare_arrays(df: pd.DataFrame):
    needed = x1_cols + x2_cols + x3_cols + x4_cols
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    X1 = df[x1_cols].to_numpy(dtype=np.float32)
    X2 = df[x2_cols].to_numpy(dtype=np.float32)
    X3 = df[x3_cols].to_numpy(dtype=np.float32)
    X4 = df[x4_cols].to_numpy(dtype=np.float32)
    return X1, X2, X3, X4

class QuadDataset(Dataset):
    def __init__(self, X1, X2, X3, X4):
        self.X1 = torch.from_numpy(X1).float()
        self.X2 = torch.from_numpy(X2).float()
        self.X3 = torch.from_numpy(X3).float()
        self.X4 = torch.from_numpy(X4).float()

    def __len__(self):
        return self.X1.shape[0]

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.X4[idx]

import os
import json
import pandas as pd
from datetime import datetime


def save_splits_csv(
    *,
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    df_test: pd.DataFrame,
    out_dir: str,
    seed: int = None,
    note: str = "",
):
    """
    Сохраняет train / val / test в CSV + metadata.json
    """
    os.makedirs(out_dir, exist_ok=True)

    # --- сами данные ---
    df_train.to_csv(f"{out_dir}/train.csv", index=False)
    df_val.to_csv(f"{out_dir}/val.csv", index=False)
    df_test.to_csv(f"{out_dir}/test.csv", index=False)

    # --- метаданные ---
    meta = {
        "created_at": datetime.utcnow().isoformat(),
        "n_train": len(df_train),
        "n_val": len(df_val),
        "n_test": len(df_test),
        "seed": seed,
        "note": note,
        "columns": list(df_train.columns),
    }

    with open(f"{out_dir}/metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(
        f"Saved splits to '{out_dir}': "
        f"train={len(df_train)}, val={len(df_val)}, test={len(df_test)}"
    )

def load_splits_csv(out_dir: str):
    """
    Загружает train / val / test + metadata.json
    """
    df_train = pd.read_csv(f"{out_dir}/train.csv")
    df_val   = pd.read_csv(f"{out_dir}/val.csv")
    df_test  = pd.read_csv(f"{out_dir}/test.csv")

    meta = None
    meta_path = f"{out_dir}/metadata.json"
    if os.path.exists(meta_path):
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

    return {
        "df_train": df_train,
        "df_val": df_val,
        "df_test": df_test,
        "metadata": meta,
    }

from sklearn.model_selection import train_test_split



X_COLS = ["x1_participant_J", "x1_participant_F", "x1_participant_Y"]
Y_COLS = ["x1_location_0", "x1_location_1", "x1_location_2", "x1_location_3"]


def main():
    df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref4feat16.csv')

    missing = [c for c in (X_COLS + Y_COLS) if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")

    X = df[X_COLS].to_numpy(dtype=np.float32)          # (N,3)
    Y_onehot = df[Y_COLS].to_numpy(dtype=np.float32)   # (N,4)
    y = Y_onehot.argmax(axis=1).astype(np.int64)       # (N,)

    # sanity: targets one-hot?
    bad_rows = float(np.mean(np.abs(Y_onehot.sum(axis=1) - 1.0) > 1e-6))
    if bad_rows > 0:
        print(f"[WARN] Non one-hot targets: {bad_rows:.2%} rows")

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=0,
        #stratify=y if len(np.unique(y)) > 1 else None
    )

    # baseline: always predict most frequent class from TRAIN
    # 1) Global majority (как было)
    vals, counts = np.unique(y_tr, return_counts=True)
    majority_class = int(vals[counts.argmax()])
    baseline_global_acc = float((y_te == majority_class).mean())

    # 2) Participant-conditional majority baseline
    # participant index: 0=J, 1=F, 2=Y
    p_tr = X_tr.argmax(axis=1)  # (N_train,) because X is one-hot
    p_te = X_te.argmax(axis=1)

    # for each participant, find most frequent class in TRAIN
    majority_by_p = {}
    for p in (0, 1, 2):
        mask = (p_tr == p)
        if mask.sum() == 0:
            # если вдруг в train нет такого участника (редко), fallback на global majority
            majority_by_p[p] = majority_class
        else:
            vals_p, cnts_p = np.unique(y_tr[mask], return_counts=True)
            majority_by_p[p] = int(vals_p[cnts_p.argmax()])

    # predict on TEST using mapping
    y_pred_baseline_p = np.array([majority_by_p[p] for p in p_te], dtype=np.int64)
    baseline_participant_acc = float((y_pred_baseline_p == y_te).mean())

    # pretty print mapping
    p_names = {0: "J", 1: "F", 2: "Y"}
    print("Baseline mapping (participant -> most popular location in TRAIN):")
    for p in (0, 1, 2):
        print(f"  participant_{p_names[p]} -> location_{majority_by_p[p]}")

    print(f"Baseline test acc (global majority): {baseline_global_acc:.3f}")
    print(f"Baseline test acc (per-participant majority): {baseline_participant_acc:.3f}")

    # torch tensors (CPU достаточно)
    device = "cpu"
    X_tr_t = torch.from_numpy(X_tr).to(device)
    y_tr_t = torch.from_numpy(y_tr).to(device)
    X_te_t = torch.from_numpy(X_te).to(device)
    y_te_t = torch.from_numpy(y_te).to(device)

    # tiny MLP: 1 hidden layer
    class TinyMLP(nn.Module):
        def __init__(self, in_dim=3, hidden=8, out_dim=4):
            super().__init__()
            self.fc1 = nn.Linear(in_dim, hidden)
            self.fc2 = nn.Linear(hidden, out_dim)
        def forward(self, x):
            return self.fc2(F.relu(self.fc1(x)))

    model = TinyMLP(in_dim=3, hidden=8, out_dim=4).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)

    def eval_acc(Xt, yt):
        model.eval()
        with torch.no_grad():
            logits = model(Xt)
            loss = F.cross_entropy(logits, yt).item()
            acc = (logits.argmax(dim=1) == yt).float().mean().item()
        return loss, acc

    # train (full-batch — очень быстро, данных мало)
    epochs = 500
    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(X_tr_t)
        loss = F.cross_entropy(logits, y_tr_t)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if epoch % 10==0:
            tr_loss, tr_acc = eval_acc(X_tr_t, y_tr_t)
            te_loss, te_acc = eval_acc(X_te_t, y_te_t)
            print(f"epoch {epoch:03d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | test loss {te_loss:.4f} acc {te_acc:.3f}")

    te_loss, te_acc = eval_acc(X_te_t, y_te_t)

    # distribution
    target_counts = df[Y_COLS].sum(axis=0).astype(int).to_dict()

    print("\n--- Summary ---")
    print("Target counts (overall):", target_counts)
    print(f"Majority class (train): location_{majority_class}")
    print(f"TinyMLP test acc: {te_acc:.3f} (test loss {te_loss:.4f})")

In [15]:
#if __name__ == "__main__":
#    main()
main()

Baseline mapping (participant -> most popular location in TRAIN):
  participant_J -> location_1
  participant_F -> location_1
  participant_Y -> location_2
Baseline test acc (global majority): 0.448
Baseline test acc (per-participant majority): 0.470
epoch 010 | train loss 1.2768 acc 0.477 | test loss 1.2806 acc 0.470
epoch 020 | train loss 1.1999 acc 0.477 | test loss 1.2055 acc 0.470
epoch 030 | train loss 1.1522 acc 0.477 | test loss 1.1583 acc 0.470
epoch 040 | train loss 1.1340 acc 0.451 | test loss 1.1387 acc 0.448
epoch 050 | train loss 1.1268 acc 0.477 | test loss 1.1295 acc 0.470
epoch 060 | train loss 1.1229 acc 0.477 | test loss 1.1240 acc 0.470
epoch 070 | train loss 1.1213 acc 0.477 | test loss 1.1219 acc 0.470
epoch 080 | train loss 1.1201 acc 0.477 | test loss 1.1210 acc 0.470
epoch 090 | train loss 1.1189 acc 0.477 | test loss 1.1201 acc 0.470
epoch 100 | train loss 1.1180 acc 0.477 | test loss 1.1195 acc 0.470
epoch 110 | train loss 1.1172 acc 0.477 | test loss 1.1193 

In [11]:

# ----- твои колонки -----
X_COLS = ["x1_participant_J", "x1_participant_F", "x1_participant_Y"]
Y_COLS = ["x1_location_0", "x1_location_1", "x1_location_2", "x1_location_3"]
P_NAMES = {0: "J", 1: "F", 2: "Y"}


# ----- утилиты -----
def df_to_xy(df: pd.DataFrame):
    missing = [c for c in (X_COLS + Y_COLS) if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    X = df[X_COLS].to_numpy(dtype=np.float32)         # (N,3)
    Y_onehot = df[Y_COLS].to_numpy(dtype=np.float32)  # (N,4)
    y = Y_onehot.argmax(axis=1).astype(np.int64)      # (N,)

    # sanity: one-hot?
    bad_rows = float(np.mean(np.abs(Y_onehot.sum(axis=1) - 1.0) > 1e-6))
    return X, y, bad_rows


def baseline_per_participant(X_tr, y_tr, X_te, y_te):
    """
    Baseline: для каждого participant (J/F/Y) берем самый популярный location на TRAIN.
    Потом на TEST предсказываем согласно participant.
    """
    # глобальный majority
    vals, counts = np.unique(y_tr, return_counts=True)
    global_major = int(vals[counts.argmax()])
    acc_global = float((y_te == global_major).mean())

    # participant index (0/1/2) — берём argmax, т.к. one-hot
    p_tr = X_tr.argmax(axis=1)
    p_te = X_te.argmax(axis=1)

    major_by_p = {}
    for p in (0, 1, 2):
        mask = (p_tr == p)
        if mask.sum() == 0:
            major_by_p[p] = global_major
        else:
            vals_p, cnts_p = np.unique(y_tr[mask], return_counts=True)
            major_by_p[p] = int(vals_p[cnts_p.argmax()])

    y_pred = np.array([major_by_p[p] for p in p_te], dtype=np.int64)
    acc_pp = float((y_pred == y_te).mean())

    return {
        "global_majority_class": global_major,
        "acc_global": acc_global,
        "majority_by_participant": major_by_p,
        "acc_per_participant": acc_pp,
    }


# ----- tiny MLP -----
class TinyMLP(nn.Module):
    def __init__(self, in_dim=3, hidden=8, out_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


@torch.no_grad()
def eval_split(model, X_t, y_t):
    model.eval()
    logits = model(X_t)
    loss = F.cross_entropy(logits, y_t).item()
    acc = (logits.argmax(dim=1) == y_t).float().mean().item()
    return loss, acc


def train_tiny_classifier_on_splits(
    splits_dir: str,
    *,
    hidden=8,
    lr=1e-2,
    weight_decay=1e-4,
    epochs=200,
    patience=30,          # early stopping по val loss
    device=None
):
    # 1) load splits
    splits = load_splits_csv(splits_dir)
    df_tr, df_va, df_te = splits["df_train"], splits["df_val"], splits["df_test"]

    # 2) make X/y
    X_tr, y_tr, bad_tr = df_to_xy(df_tr)
    X_va, y_va, bad_va = df_to_xy(df_va)
    X_te, y_te, bad_te = df_to_xy(df_te)

    if bad_tr or bad_va or bad_te:
        print(f"[WARN] Non one-hot targets: train={bad_tr:.2%}, val={bad_va:.2%}, test={bad_te:.2%}")

    # 3) baselines (на val и test — отдельно, но mapping учим ТОЛЬКО на train!)
    base_val = baseline_per_participant(X_tr, y_tr, X_va, y_va)
    base_test = baseline_per_participant(X_tr, y_tr, X_te, y_te)

    print("\nBaseline mapping learned on TRAIN:")
    for p in (0, 1, 2):
        print(f"  participant_{P_NAMES[p]} -> location_{base_test['majority_by_participant'][p]}")
    print(f"VAL  baseline acc: global={base_val['acc_global']:.3f}, per-participant={base_val['acc_per_participant']:.3f}")
    print(f"TEST baseline acc: global={base_test['acc_global']:.3f}, per-participant={base_test['acc_per_participant']:.3f}")

    # 4) torch tensors
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_tr_t = torch.from_numpy(X_tr).to(device)
    y_tr_t = torch.from_numpy(y_tr).to(device)
    X_va_t = torch.from_numpy(X_va).to(device)
    y_va_t = torch.from_numpy(y_va).to(device)
    X_te_t = torch.from_numpy(X_te).to(device)
    y_te_t = torch.from_numpy(y_te).to(device)

    # 5) model + opt
    model = TinyMLP(in_dim=3, hidden=hidden, out_dim=4).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 6) train with best-on-val
    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(X_tr_t)
        loss = F.cross_entropy(logits, y_tr_t)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        val_loss, val_acc = eval_split(model, X_va_t, y_va_t)

        improved = val_loss < best_val_loss - 1e-6
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch in (1, 5, 10, 20, 50, 100, epochs):
            tr_loss, tr_acc = eval_split(model, X_tr_t, y_tr_t)
            print(f"epoch {epoch:03d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")

        if patience is not None and no_improve >= patience:
            print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}, best val loss {best_val_loss:.4f})")
            break

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} (val loss {best_val_loss:.4f})")

    # final test
    test_loss, test_acc = eval_split(model, X_te_t, y_te_t)
    print(f"\nTinyMLP TEST: loss {test_loss:.4f} acc {test_acc:.3f}")

    return model, {
        "device": device,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "baseline_val": base_val,
        "baseline_test": base_test,
        "test_loss": test_loss,
        "test_acc": test_acc,
    }

In [12]:
#/content/drive/MyDrive/monkeytype/

model, info = train_tiny_classifier_on_splits(
    splits_dir="/content/drive/MyDrive/monkeytype/splits_v1",   # папка где train.csv/val.csv/test.csv
    hidden=8,
    lr=1e-2,
    epochs=200,
    patience=30
)
print(info["test_acc"], info["baseline_test"]["acc_per_participant"])


Baseline mapping learned on TRAIN:
  participant_J -> location_1
  participant_F -> location_1
  participant_Y -> location_2
VAL  baseline acc: global=0.453, per-participant=0.476
TEST baseline acc: global=0.448, per-participant=0.470
epoch 001 | train loss 1.4127 acc 0.450 | val loss 1.4043 acc 0.453
epoch 005 | train loss 1.3420 acc 0.450 | val loss 1.3347 acc 0.453
epoch 010 | train loss 1.2765 acc 0.450 | val loss 1.2707 acc 0.453
epoch 020 | train loss 1.2048 acc 0.450 | val loss 1.2032 acc 0.453
epoch 050 | train loss 1.1332 acc 0.477 | val loss 1.1430 acc 0.476
epoch 100 | train loss 1.1181 acc 0.477 | val loss 1.1322 acc 0.476
Early stopping at epoch 199 (best epoch 169, best val loss 1.1294)
Restored best model from epoch 169 (val loss 1.1294)

TinyMLP TEST: loss 1.1215 acc 0.470
0.4695187211036682 0.46951871657754013
